![wandb-logo](http://wandb.me/logo-im-png)
<!--- @wandbcode{what-is-torch-nn} -->

In [ ]:
%%capture

###
#  Setup for presentation version
###

import os
from pathlib import Path

if os.name == "nt":  # resolve windows path issue with graphviz
    gv_path = str(Path("c:") / "Program Files" / "Graphviz" / "bin")
    path = %env PATH
    path = path if path.endswith(gv_path + ";" ) else path + gv_path
    %env path={path}
else:
    !apt install graphviz

!pip install torchviz wandb
import wandb

nb_path = Path(os.getcwd()) / "nn_tutorial++.ipynb"
%env WANDB_NOTEBOOK_NAME={nb_path}

def _repr_mimebundle_(  # over-write for better look of Run Dashboard in RISE slides
        self, include=None, exclude=None
    ):
        url = self._get_run_url()
        style = "border:none;width:100%;height:720px"
        s = '<h1>Run({})</h1><iframe src="{}" style="{}"></iframe>'.format(
            self._run_id, url, style
        )
        return {"text/html": s}
    
wandb.sdk.wandb_run.Run._repr_mimebundle_ = _repr_mimebundle_


What is `torch.nn` *really*?
============================

by Jeremy Howard, [fast.ai](https://www.fast.ai).



Modified for use with [RISE](https://rise.readthedocs.io/en/stable/)
by [Charles Frye](https://charlesfrye.github.io).

PyTorch provides the elegantly designed modules and classes
[`torch.nn`](https://pytorch.org/docs/stable/nn.html),
[`torch.optim`](https://pytorch.org/docs/stable/optim.html) ,
[`Dataset`](https://pytorch.org/docs/stable/data.html?highlight=dataset#torch.utils.data.Dataset),
and [`DataLoader`](https://pytorch.org/docs/stable/data.html?highlight=dataloader#torch.utils.data.DataLoader)
to help you create and train neural networks.

In order to fully utilize their power and customize
them for your problem, you need to really understand exactly what they're
doing.

To develop this understanding, we will first train basic neural net
on the MNIST data set without using any features from these models; we will
initially only use the most basic PyTorch tensor functionality.

Then, we will incrementally add one feature from ``torch.nn``, ``torch.optim``, ``Dataset``, or
``DataLoader`` at a time, showing exactly what each piece does, and how it
works to make the code either more concise, or more flexible.

**This tutorial assumes you already have PyTorch installed, and are familiar
with the basics of tensor operations.** (If you're familiar with Numpy array
operations, you'll find the PyTorch tensor operations used here nearly identical).

MNIST data setup
----------------

We will use the classic [`MNIST`](<http://deeplearning.net/data/mnist) dataset,
which consists of black-and-white images of hand-drawn digits (between 0 and 9).

We will use [`pathlib`](https://docs.python.org/3/library/pathlib.html)
for dealing with paths (part of the Python 3 standard library),

and will download the dataset using
[`requests`](http://docs.python-requests.org/en/master/).

In [ ]:
from pathlib import Path
import requests

We will only
import modules when we use them, so you can see exactly what's being
used at each point.

In [ ]:
DATA_PATH = Path("data")
PATH = DATA_PATH / "mnist"

PATH.mkdir(parents=True, exist_ok=True)

In [ ]:
URL = "https://github.com/pytorch/tutorials/raw/master/_static/"
FILENAME = "mnist.pkl.gz"

if not (PATH / FILENAME).exists():
        content = requests.get(URL + FILENAME).content
        (PATH / FILENAME).open("wb").write(content)

This dataset is in numpy array format, and has been stored using pickle,
a python-specific format for serializing data.



In [ ]:
import pickle
import gzip

with gzip.open((PATH / FILENAME).as_posix(), "rb") as f:
        ((x_train, y_train), (x_valid, y_valid), _) = pickle.load(f, encoding="latin-1")

Each image is 28 x 28, and is being stored as a flattened row of length
784 (=28x28).

Let's take a look at one; we need to reshape it to 2d first.

In [ ]:
import numpy as np
import wandb  # logging library -- we'll use it more later!

print(x_train.shape)
im = wandb.Image(x_train[0].reshape((28, 28)))
im.image

PyTorch uses `torch.tensor`s, rather than `np.ndarray`s, so we need to
convert our data.

In [ ]:
import torch

x_train, y_train, x_valid, y_valid = map(torch.tensor,
                                         (x_train, y_train, x_valid, y_valid))
n, c = x_train.shape
print(x_train, y_train)
print(x_train.shape)
print(y_train.min(), y_train.max())

Neural net from scratch (no `torch.nn`)
---------------------------------------------

Let's first create a model using nothing but PyTorch tensor operations.

We're assuming you're already familiar with the basics of neural networks.

(If you're not, you can
learn them at [course.fast.ai](https://course.fast.ai)).

PyTorch provides methods to create random or zero-filled tensors, which we will
use to create our weights and biases (😏) for a simple linear model.

These are just regular tensors, with one very special addition:

we tell PyTorch that they require a gradient.

This causes PyTorch to record all of the operations done on the tensor,
so that it can calculate the gradient during back-propagation *automatically*!

For the weights, we set `requires_grad` **after** the initialization, since we
don't want that step included in the gradient.

In [ ]:
import math

weights = torch.randn(784, 10) / math.sqrt(784)
weights.requires_grad_()
biases = torch.zeros(10, requires_grad=True)

(Note that a trailing `_` in
PyTorch signifies that the operation is performed in-place.)

<div class="alert alert-info"><h4>Note</h4><p>
    We are initializing the weights here with
    <a href=http://proceedings.mlr.press/v9/glorot10a/glorot10a.pdf>Xavier initialisation</a>,
    i.e. by multiplying with 1/sqrt(n).</p></div>

Thanks to PyTorch's ability to calculate gradients automatically, we can
use any standard Python function (or callable object) as a model!

So let's just write a plain matrix multiplication and broadcasted addition
to create a simple linear model. We also need an activation function, so
we'll write `log_softmax` and use it.

In [ ]:
def log_softmax(x):
    return x - x.exp().sum(-1).log().unsqueeze(-1)

def model(xb):
    return log_softmax(xb @ weights + biases)

In the above, the `@` stands for the dot product operation.

> Remember: although PyTorch provides lots of pre-written loss functions, activation functions, and so forth, you can easily write your own using plain Python. PyTorch will
even create fast GPU or vectorized CPU code for your function
automatically.

We will call
our function on one batch of data (in this case, 64 images).

This is one *forward pass*.  Note that our predictions won't be any better than
random at this stage, since we start with random weights.

In [ ]:
bs = 64  # batch size

xb = x_train[0:bs]  # a mini-batch from x
preds = model(xb)  # predictions
preds[0], preds.shape
print(preds[0], preds.shape) 

As you see, the `preds` tensor contains not only the tensor values, but also a
gradient function (`grad_fn`). We'll use this later to do backprop.

In [ ]:
import torchviz

torchviz.make_dot(preds, params={"W": weights, "b": biases})

Let's implement [negative log-likelihood](https://charlesfrye.github.io/stats/2017/11/09/the-surprise-game.html)
to use as the loss function
(again, we can just use standard Python):

In [ ]:
def nll(input, target):
    nlls = -input[range(target.shape[0]), target]  # hack for log-likelihood with integer labels
    return nlls.mean()

loss_func = nll

Let's check our loss with our random model, so we can see if we improve
after a backprop pass later.



In [ ]:
yb = y_train[0:bs]
starting_loss = loss_func(preds, yb)
print(starting_loss, np.log(10))

Let's also implement a function to calculate the accuracy of our model.

For each prediction, if the index with the largest value matches the
target value, then the prediction was correct.

In [ ]:
def accuracy(out, yb):
    preds = torch.argmax(out, dim=1)
    is_correct = preds == yb
    return is_correct.float().mean()

Let's check the accuracy of our random model, so we can see if our
accuracy improves as our loss improves.



In [ ]:
starting_acc = accuracy(preds, yb)
print(starting_acc)

We can now run a training loop. For each iteration, we will:

1. Select a mini-batch of data of size `bs` (`L7`)
2. Use the `model` to make `pred`ictions (`L9`)
3. Calculate the `loss` on the batch (`L10`)
4. Use `loss.backward()` to update the `grad`ients of the `model` (`L12`)
5. Update the parameters of the model using the `grad`ients (`L15` and `L15`)

In [ ]:
lr = 0.5  # learning rate
epochs = 2  # how many epochs to train for

for epoch in range(epochs):
    for ii in range((n - 1) // bs + 1):
        start, end = ii * bs, ii * bs + bs
        xb, yb = x_train[start:end], y_train[start:end]

        pred = model(xb)
        loss = loss_func(pred, yb)
        
        loss.backward()
        with torch.no_grad():
            weights -= weights.grad * lr
            biases -= biases.grad * lr
            weights.grad.zero_()
            biases.grad.zero_()

We do the update within the `torch.no_grad()` context manager, because we do not want these
actions to be recorded for our next calculation of the gradient.

You can read more about how PyTorch's Autograd records operations
[here](https://pytorch.org/docs/stable/notes/autograd.html).

We then set the gradients to zero, so that we are ready for the next loop.

Otherwise, our gradients would record a running tally of all the operations
that had happened (i.e. ``loss.backward()`` *adds* the gradients to whatever is
already stored, rather than replacing them).

That's it: we've created and trained a minimal neural network (in this case, a
logistic regression, since we have no hidden layers) entirely from scratch!

Let's check the loss and accuracy and compare those to what we got
earlier. We expect that the loss will have decreased and accuracy to
have increased, and they have.



In [ ]:
print(starting_loss, starting_acc)
print(loss_func(model(xb), yb), accuracy(model(xb), yb))

Refactor using `torch.nn.functional`
------------------------------

We will now refactor our code, so that it does the same thing as before, only
we'll start taking advantage of PyTorch's `nn` classes to make it more concise
and flexible.

At each step from here, we should be making our code one or more
of: shorter, more understandable, and/or more flexible.

The first and easiest step is to make our code shorter by replacing our
hand-written activation and loss functions with those from `torch.nn.functional`
(which is generally imported into the namespace `F` by convention).

In [ ]:
import torch.nn.functional as F

[elem for elem in dir(F) if not elem.startswith("_")]

This module contains all the functions in the `torch.nn` library
(whereas other parts of the library contain classes).
As well as a wide range of loss and activation
functions, you'll also find here some convenient functions for creating neural
nets, such as pooling functions.

(There are also functions for doing convolutions,
linear operations, etc, but as we'll see, these are usually better handled using
other parts of the library.)

If you're using negative log likelihood loss and log softmax activation,
then Pytorch provides a single function `F.cross_entropy` that combines
the two. So we can even remove the activation function from our model.

In [ ]:
loss_func = F.cross_entropy

def model(xb):
    return xb @ weights + biases

Note that we no longer call ``log_softmax`` in the ``model`` function. Let's
confirm that our loss and accuracy are the same as before:



In [ ]:
print(loss_func(model(xb), yb), accuracy(model(xb), yb))

Refactor using `nn.Module`
-----------------------------

Next up, we'll use ``nn.Module`` and ``nn.Parameter``, for a clearer and more
concise training loop.

In [ ]:
from torch import nn

We subclass ``nn.Module`` (which itself is a class and
able to keep track of state).  In this case, we want to create a class that
holds our weights, biases, and method for the forward step.

`nn.Module` has a
number of attributes and methods (such as `.parameters()` and `.zero_grad()`),
which we will be using.

In [ ]:
class Mnist_Logistic(nn.Module):
    def __init__(self):
        super().__init__()
        self.weights = nn.Parameter(torch.randn(784, 10) / math.sqrt(784))
        self.biases = nn.Parameter(torch.zeros(10))

    def forward(self, xb):
        return xb @ self.weights + self.biases

> Note: `nn.Module` (uppercase M) is a PyTorch specific concept, and is a
class we'll be using a lot. `nn.Module` is not to be confused with the Python
concept of a (lowercase `m`) [`module`](https://docs.python.org/3/tutorial/modules.html),
which is a file of Python code that can be imported.

Since we're now using an object instead of just using a function, we
first have to instantiate our model:



In [ ]:
model = Mnist_Logistic()

Now we can calculate the loss in the same way as before. Note that
``nn.Module`` objects are used as if they are functions (i.e they are
*callable*), but behind the scenes Pytorch will call our ``forward``
method automatically.



In [ ]:
print(loss_func(model(xb), yb))

Previously for our training loop we had to update the values for each parameter
by name, and manually zero out the grads for each parameter separately, like this:
```python
with torch.no_grad():
    weights -= weights.grad * lr
    biases -= biases.grad * lr
    weights.grad.zero_()
    biases.grad.zero_()
```

Now we can take advantage of `model.parameters()` and `model.zero_grad()`
(which are both defined by PyTorch for ``nn.Module``)
to make those steps more concise
and less prone to the error of forgetting some of our parameters, particularly
if we had a more complicated model:

```python
with torch.no_grad():
    for p in model.parameters():
        p -= p.grad * lr
    model.zero_grad()
```

We'll wrap our little training loop in a ``fit`` function so we can run it
again later.

In [ ]:
def fit():
    for epoch in range(epochs):
        for i in range((n - 1) // bs + 1):
            start_i = i * bs
            end_i = start_i + bs
            xb = x_train[start_i:end_i]
            yb = y_train[start_i:end_i]
            pred = model(xb)
            loss = loss_func(pred, yb)

            loss.backward()
            with torch.no_grad():
                for p in model.parameters():
                    p -= p.grad * lr
                model.zero_grad()

fit()

Let's double-check that our loss has gone down:



In [ ]:
print(loss_func(model(xb), yb))

Refactor using `nn.Linear`
-------------------------

We continue to refactor our code.

Instead of manually defining and
initializing `self.weights` and `self.biases`, and calculating `xb  @ self.weights + self.biases`, we will instead use the Pytorch class
[`nn.Linear`](https://pytorch.org/docs/stable/nn.html#linear-layers>) for a
linear layer, which does all that for us.

In [ ]:
nn.Linear??

Pytorch has many types of
predefined layers that can greatly simplify our code, and often makes it
faster too.

In [ ]:
class Mnist_Logistic(nn.Module):
    def __init__(self):
        super().__init__()
        self.lin = nn.Linear(in_features=784, out_features=10)

    def forward(self, xb):
        return self.lin(xb)

We instantiate our model and calculate the loss in the same way as before:



In [ ]:
model = Mnist_Logistic()
print(loss_func(model(xb), yb))

We are still able to use our same ``fit`` method as before.



In [ ]:
fit()

print(loss_func(model(xb), yb))

Refactor using `optim`
------------------------------

In [ ]:
from torch import optim

Pytorch also has a package with various optimization algorithms, `torch.optim`.
We can use the ``step`` method from our optimizer to update the parameters, instead
of doing it manually for each parameter.

This will let us replace our previous manually coded optimization step:
```python
with torch.no_grad():
    for p in model.parameters():
        p -= p.grad * lr
    model.zero_grad()
```
and instead use just:
```python
opt.step()
opt.zero_grad()
```

(``optim.zero_grad()`` resets the gradient to `0` and we need to call it before
computing the gradient for the next minibatch.)

We'll define a little function to create our model and optimizer so we
can reuse it in the future.



In [ ]:
def get_model():
    model = Mnist_Logistic()
    return model, optim.SGD(model.parameters(), lr=lr)

In [ ]:
model, opt = get_model()
print(loss_func(model(xb), yb))

for epoch in range(epochs):
    for i in range((n - 1) // bs + 1):
        start_i = i * bs
        end_i = start_i + bs
        xb = x_train[start_i:end_i]
        yb = y_train[start_i:end_i]
        pred = model(xb)
        loss = loss_func(pred, yb)

        loss.backward()
        opt.step()
        opt.zero_grad()

print(loss_func(model(xb), yb))

Refactor using `Dataset`
------------------------------

In [ ]:
from torch.utils.data import TensorDataset

PyTorch has an abstract `Dataset` class.

A `Dataset` can be anything that has
a `__len__` function (called by Python's standard `len` function) and
a `__getitem__` function as a way of indexing into it.

[This tutorial](https://pytorch.org/tutorials/beginner/data_loading_tutorial.html)
walks through a nice example of creating a custom `FacialLandmarkDataset` class
as a subclass of `Dataset`.

PyTorch's [`TensorDataset`](https://pytorch.org/docs/stable/_modules/torch/utils/data/dataset.html#TensorDataset)
is a `Dataset` wrapping `Tensor`s.

By defining a length and way of indexing,
this also gives us a way to iterate, index, and slice along the first
dimension of a tensor. This will make it easier to access both the
independent and dependent variables in the same line as we train.

Both ``x_train`` and ``y_train`` can be combined in a single ``TensorDataset``,
which will be easier to iterate over and slice.



In [ ]:
train_ds = TensorDataset(x_train, y_train)

Previously, we had to iterate through minibatches of x and y values separately:
```python
xb = x_train[start:end]
yb = y_train[start:end]
```

Now, we can do these two steps together:
```python
xb, yb = train_ds[start:end]
```

In [ ]:
model, opt = get_model()

for epoch in range(epochs):
    for i in range((n - 1) // bs + 1):
        start, end = i * bs, i * bs + bs
        xb, yb = train_ds[start:end]
        pred = model(xb)
        loss = loss_func(pred, yb)

        loss.backward()
        opt.step()
        opt.zero_grad()

print(loss_func(model(xb), yb))

Refactor using `DataLoader`
------------------------------

In [ ]:
from torch.utils.data import DataLoader

Pytorch's `DataLoader` is responsible for managing batches. You can
create a `DataLoader` from any `Dataset`.

`DataLoader` makes it easier to iterate over batches.

Rather than having to use `train_ds[i*bs:i*bs+bs]`,
the `DataLoader` gives us each minibatch automatically.

In [ ]:
train_ds = TensorDataset(x_train, y_train)
train_dl = DataLoader(train_ds, batch_size=bs)

Previously, our loop iterated over batches `(xb, yb)` like this:
```python
for i in range((n-1)//bs + 1):
    xb, yb = train_ds[i*bs : i*bs+bs]
    pred = model(xb)
```

Now, our loop is much cleaner, as `(xb, yb)` are loaded automatically from the data loader:
```python
for xb, yb in train_dl:
    pred = model(xb)
```

In [ ]:
model, opt = get_model()

for epoch in range(epochs):
    for xb, yb in train_dl:
        pred = model(xb)
        loss = loss_func(pred, yb)

        loss.backward()
        opt.step()
        opt.zero_grad()

print(loss_func(model(xb), yb))

## So what is `torch.nn`, really?

   + `Module`: creates a callable which behaves like a function, but can also
     contain state (such as neural net layer weights). It is responsible for the `Parameter`(s) it
     contains.

   + `Parameter`: a wrapper for a tensor that tells a `Module` that it has weights (and biases 😏)
     that need updating during backprop.

   + `functional`: a module (imported as `F` by convention)
     which contains non-stateful operations, like losses, activations, and matrix multiplication

 - `torch.optim`: Contains optimizers, such as `SGD`, which update the weights
   of `Parameter` during the backward step

 - `Dataset`: An abstract interface for data, based on `__len__` and `__getitem__`

 - `DataLoader`: Takes any `Dataset` and creates an iterator which returns batches of data.

# 🎉

Thanks to Pytorch's `nn.Module`, `nn.Parameter`, `Dataset`, and `DataLoader`,
our training loop is now dramatically smaller and easier to understand.

Let's now try to add the basic features necessary to create effective models in practice.

Add logging
-----------------------

![logo](http://wandb.me/logo-im-png)

Weights & Biases is a developer toolkit for "multiplayer" machine learning,
with components ranging from logging tools and model registration to interactive dashboards and job scheduling.

In [ ]:
import wandb
    #  double-you-and-bee: W&🅱️
    # or wand-bee: ✨🐝 
    # or wan-DB: 📁

In [ ]:
model, opt = get_model()

run = wandb.init(project="torch-nn")

for epoch in range(epochs):
    for xb, yb in train_dl:
        pred = model(xb)
        loss = loss_func(pred, yb)

        loss.backward()
        opt.step()
        opt.zero_grad()
        
        wandb.log({"loss": loss, "epoch": epoch}) 

We can view most information logged to W&B directly inside Jupyter:

In [ ]:
run

In [ ]:
run.finish()  # close the run out and get a summary

In [our PyTorch integration](https://docs.wandb.ai/guides/integrations/pytorch),
we have a special tool for tracking gradients and graphs --
[`wandb.watch`](https://docs.wandb.ai/ref/python/watch).

In [ ]:
model, opt = get_model()

run = wandb.init(project="torch-nn")
wandb.watch(model, log="all",   # "watch" values of gradients and parameters
            log_freq=100,  # log them every 100 steps
            log_graph=True)  # log the compute graph

for epoch in range(epochs):
    for xb, yb in train_dl:
        pred = model(xb)
        loss = loss_func(pred, yb)

        loss.backward()
        opt.step()
        opt.zero_grad()
        
        wandb.log({"loss": loss, "epoch": epoch})
        
run.finish()

# Learn more about using Weights & Biases!

In [ ]:
from IPython.display import YouTubeVideo

In [ ]:
pytorch_video_id = "G7GH0SeNBMA"
learn_wandb_playlist = "PLD80i8An1OEGajeVo15ohAQYF1Ttle0lk"
YouTubeVideo(pytorch_video_id, list=learn_wandb_playlist, listType="playlist", rel=0, width=756, height=640)

Add validation
-----------------------

In section 1, we were just trying to get a reasonable training loop set up for
use on our training data.

In reality, you **always** should also have
a [validation set](https://www.fast.ai/2017/11/13/validation-sets/), in order
to identify if you are overfitting.

In [ ]:
train_ds = TensorDataset(x_train, y_train)
train_dl = DataLoader(train_ds, batch_size=bs, shuffle=True)

valid_ds = TensorDataset(x_valid, y_valid)
valid_dl = DataLoader(valid_ds, batch_size=bs * 2)

Shuffling the training data is
[important](https://www.quora.com/Does-the-order-of-training-data-matter-when-training-neural-networks)
to prevent correlation between batches and overfitting.

On the other hand, the validation loss will be identical whether we shuffle the validation set or not.

We'll use a batch size for the validation set that is twice as large as
that for the training set.

This is because the validation set does not
need backpropagation and thus takes less memory (it doesn't need to store the gradients).

We take advantage of this to use a larger batch size and compute the loss more quickly.

We will calculate and log the validation loss at the end of each epoch.

(Note that we always call ``model.train()`` before training, and ``model.eval()``
before inference, because these are used by layers such as ``nn.BatchNorm2d``
and ``nn.Dropout`` to ensure appropriate behaviour for these different phases.)

In [ ]:
model, opt = get_model()

run = wandb.init(project="torch-nn")
wandb.watch(model, log="all", log_freq=100, log_graph=True)

for epoch in range(epochs):
    model.train()
    for xb, yb in train_dl:
        pred = model(xb)
        loss = loss_func(pred, yb)

        loss.backward()
        opt.step()
        opt.zero_grad()
        
        wandb.log({"train/loss": loss, "epoch": epoch})

    model.eval()
    with torch.no_grad():
        valid_loss = sum(loss_func(model(xb), yb) for xb, yb in valid_dl)

    wandb.log({"val/loss": loss, "epoch": epoch})
    
run.finish()

Create `fit()` and `get_data()`
----------------------------------

We'll now do a little refactoring of our own. Since we go through a similar
process twice of calculating the loss for both the training set and the
validation set, let's make that into its own function, ``loss_batch``, which
computes the loss for one batch.

We pass an optimizer in for the training set, and use it to perform
backprop.  For the validation set, we don't pass an optimizer, so the
method doesn't perform backprop.

In [ ]:
def loss_batch(model, loss_func, xb, yb, opt=None):
    loss = loss_func(model(xb), yb)

    if opt is not None:
        loss.backward()
        opt.step()
        opt.zero_grad()

    return loss.item(), len(xb)

``fit`` runs the necessary operations to train our model and compute the
training and validation losses for each epoch.



In [ ]:
import numpy as np

def fit(epochs, model, loss_func, opt, train_dl, valid_dl, on_train_step=None, on_val_epoch_end=None):
    for epoch in range(epochs):
        model.train()
        for xb, yb in train_dl:
            loss, batch_sz = loss_batch(model, loss_func, xb, yb, opt)
            on_train_step({"train/loss": loss, "train/batch_sz": batch_sz}) if on_train_step is not None else None
            

        model.eval()
        with torch.no_grad():
            losses, nums = zip(
                *[loss_batch(model, loss_func, xb, yb) for xb, yb in valid_dl]
            )
        val_loss = np.sum(np.multiply(losses, nums)) / np.sum(nums)
        on_val_epoch_end({"val/loss": val_loss}) if on_val_epoch_end is not None else None

Notice the inclusion of two "callbacks" -- places in the code where a provided function is called.

A more sophisticated version of this system is how logging code and similar "bonus features" are added in libraries like `fastai` and `pytorch_lightning`.

``get_data`` returns dataloaders for the training and validation sets.



In [ ]:
def get_data(train_ds, valid_ds, bs):
    return (
        DataLoader(train_ds, batch_size=bs, shuffle=True),
        DataLoader(valid_ds, batch_size=bs * 2),
    )

### Refactor the logging as well

In [ ]:
def log_dict(dct):
    wandb.log(dct)

In [ ]:
def fit_and_log(epochs, model, loss_func, opt, train_dl, valid_dl):
    run = wandb.init(project="torch-nn")
    wandb.watch(model, log="all", log_freq=100, log_graph=True)
    fit(epochs, model, loss_func, opt, train_dl, valid_dl, log_dict, log_dict)
    run.finish()

Now, our whole process of obtaining the data loaders and fitting the
model can be run in 3 lines of code:



In [ ]:
train_dl, valid_dl = get_data(train_ds, valid_ds, bs)  # obtain dataloaders
model, opt = get_model()  # set up model and optimizer

fit_and_log(epochs, model, loss_func, opt, train_dl, valid_dl)  # zoom!

You can use these basic 3 lines of code to train a wide variety of models.
Let's see if we can use them to train a convolutional neural network (CNN)!

Switch to `CNN`
-------------

We are now going to build our neural network with three convolutional layers.

Because none of the functions in the previous section assume anything about
the model form, we'll be able to use them to train a CNN without any modification.

We will use Pytorch's predefined
[`Conv2d`](https://pytorch.org/docs/stable/nn.html#torch.nn.Conv2d>) class
as our convolutional layer.

In [ ]:
class Mnist_CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, stride=2, padding=1)
        self.conv2 = nn.Conv2d(16, 16, kernel_size=3, stride=2, padding=1)
        self.conv3 = nn.Conv2d(16, 10, kernel_size=3, stride=2, padding=1)

    def forward(self, xb):
        xb = xb.view(-1, 1, 28, 28)
        xb = F.relu(self.conv1(xb))
        xb = F.relu(self.conv2(xb))
        xb = F.relu(self.conv3(xb))
        xb = F.avg_pool2d(xb, 4)
        return xb.view(-1, xb.size(1))

lr = 0.1

We define a CNN with 3 convolutional layers.
Each convolution is followed by a ReLU.  At the end, we perform an
average pooling.

(Note that `view` is PyTorch's version of numpy's
`reshape`)

[Momentum](https://cs231n.github.io/neural-networks-3/#sgd>) is a variation on
stochastic gradient descent that takes previous updates into account as well
and generally leads to faster training.

In [ ]:
model = Mnist_CNN()
opt = optim.SGD(model.parameters(), lr=lr, momentum=0.9)

fit_and_log(epochs, model, loss_func, opt, train_dl, valid_dl)

`nn.Sequential`
------------------------

`torch.nn` has another handy class we can use to simplify our code:
[`Sequential`](https://pytorch.org/docs/stable/nn.html#torch.nn.Sequential).

A `Sequential` object runs each of the modules contained within it, in a
sequential manner. This is a simpler way of writing our neural network.

To take advantage of this, we need to be able to easily define a
**custom layer** from a given function. 

For instance, PyTorch doesn't
have a `view` layer, and we need to create one for our network. `Lambda`
will create a layer that we can then use when defining a network with
`Sequential`.

In [ ]:
class Lambda(nn.Module):
    def __init__(self, func):
        super().__init__()
        self.func = func

    def forward(self, x):
        return self.func(x)


def preprocess(x):
    return x.view(-1, 1, 28, 28)

The model created with ``Sequential`` is simply:



In [ ]:
model = nn.Sequential(
    Lambda(preprocess),
    nn.Conv2d(1, 16, kernel_size=3, stride=2, padding=1), nn.ReLU(),
    nn.Conv2d(16, 16, kernel_size=3, stride=2, padding=1), nn.ReLU(),
    nn.Conv2d(16, 10, kernel_size=3, stride=2, padding=1), nn.ReLU(),
    nn.AvgPool2d(4),
    Lambda(lambda x: x.view(x.size(0), -1)),
)

opt = optim.SGD(model.parameters(), lr=lr, momentum=0.9)

fit_and_log(epochs, model, loss_func, opt, train_dl, valid_dl)

Wrapping `DataLoader`
-----------------------------

Our CNN is fairly concise, but it only works with MNIST, because:
 - It assumes the input is a 28\*28 long vector
 - It assumes that the final CNN grid size is 4\*4 (since that's the average
pooling kernel size we used)

Let's get rid of these two assumptions, so our model works with any 2d
single channel image. First, we can remove the initial Lambda layer by
moving the data preprocessing into a generator:

In [ ]:
def preprocess(x, y):  # need different preprocessor for different data!
    return x.view(-1, 1, 28, 28), y


class WrappedDataLoader:
    def __init__(self, dl, func):
        self.dl = dl
        self.func = func

    def __len__(self):
        return len(self.dl)

    def __iter__(self):
        batches = iter(self.dl)
        for b in batches:
            yield (self.func(*b))

train_dl, valid_dl = get_data(train_ds, valid_ds, bs)
train_dl = WrappedDataLoader(train_dl, preprocess)
valid_dl = WrappedDataLoader(valid_dl, preprocess)

Next, we can replace ``nn.AvgPool2d`` with ``nn.AdaptiveAvgPool2d``, which
allows us to define the size of the *output* tensor we want, rather than
the *input* tensor we have. As a result, our model will work with any
size input.



In [ ]:
model = nn.Sequential(
    nn.Conv2d(1, 16, kernel_size=3, stride=2, padding=1), nn.ReLU(),
    nn.Conv2d(16, 16, kernel_size=3, stride=2, padding=1), nn.ReLU(),
    nn.Conv2d(16, 10, kernel_size=3, stride=2, padding=1), nn.ReLU(),
    nn.AdaptiveAvgPool2d(1),
    Lambda(lambda x: x.view(x.size(0), -1)),
)

opt = optim.SGD(model.parameters(), lr=lr, momentum=0.9)

Let's try it out:



In [ ]:
fit_and_log(epochs, model, loss_func, opt, train_dl, valid_dl)

Using your GPU
---------------

If you're lucky enough to have access to a CUDA-capable GPU (you can
rent one for about $0.50/hour from most cloud providers) you can
use it to speed up your code.

First check that your GPU is working in
Pytorch:

In [ ]:
print(torch.cuda.is_available())

And then create a device object for it:



In [ ]:
dev = torch.device(
    "cuda") if torch.cuda.is_available() else torch.device("cpu")

Let's update ``preprocess`` to move batches to the GPU:



In [ ]:
def preprocess(x, y):
    return x.view(-1, 1, 28, 28).to(dev), y.to(dev)


train_dl, valid_dl = get_data(train_ds, valid_ds, bs)
train_dl = WrappedDataLoader(train_dl, preprocess)
valid_dl = WrappedDataLoader(valid_dl, preprocess)

Finally, we can move our model to the GPU.



In [ ]:
model = nn.Sequential(
    nn.Conv2d(1, 16, kernel_size=3, stride=2, padding=1), nn.ReLU(),
    nn.Conv2d(16, 16, kernel_size=3, stride=2, padding=1), nn.ReLU(),
    nn.Conv2d(16, 10, kernel_size=3, stride=2, padding=1), nn.ReLU(),
    nn.AdaptiveAvgPool2d(1),
    Lambda(lambda x: x.view(x.size(0), -1)),
)

opt = optim.SGD(model.parameters(), lr=lr, momentum=0.9)

model.to(dev)
opt = optim.SGD(model.parameters(), lr=lr, momentum=0.9)

You should find it runs faster now:



In [ ]:
fit_and_log(epochs, model, loss_func, opt, train_dl, valid_dl)